# DataRetail360: demostración manual de extracción, transformación y carga

Ejecuta una celda a la vez con Shift+Enter y explica cada salida. Este notebook utiliza las mismas funciones que `EJECUTAR_COMPLETO.cmd`; no ejecuta la carga completa por detrás.

Origen: siete CSV sintéticos ya existentes en `datos/entrada`. Transformación: DataFrames limpios que se guardan en Parquet. Gold vuelve a leer esos Parquet. Destino de esta demostración: `resultados/demostracion.sqlite`. Silver usa una carpeta fija con seis Parquet y su manifiesto; no se generan carpetas por ejecución. La base previa solo se reemplaza después de validar la nueva.

Antes de ensayar desde cero: cierra cualquier sesión anterior y ejecuta `LIMPIAR_RESULTADOS.cmd`. No es obligatorio limpiar para repetir: la nueva ejecución reemplaza la salida validada. Si interrumpes una celda, reinicia el kernel y comienza de nuevo.

In [ ]:
from pathlib import Path
import sys, inspect
import pandas as pd
from IPython.display import display, Code

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src' / 'proceso.py').exists() and (RAIZ.parent / 'src' / 'proceso.py').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / 'src' / 'proceso.py').exists(), 'Abre el notebook desde ABRIR_DEMOSTRACION.cmd.'
sys.path.insert(0, str(RAIZ))
from src.proceso import SesionETL
from src.extraccion.preparar import extraer_fuentes
from src.extraccion.lector_csv import leer_csv_validado
from src.transformacion.dimensiones.preparar import preparar_dimensiones
from src.transformacion.ventas import preparar_ventas
from src.transformacion.devoluciones import preparar_devoluciones
from src.carga.gold.conexion import crear_motor
from src.carga.gold.modelo import crear_modelo, ARCHIVOS_MODELO
from src.carga.gold.insercion import cargar_tablas
from src.carga.silver import guardar_silver, leer_silver

# Si ejecutaste antes la preparación en este kernel, se cancela de forma segura.
if 'sesion' in globals():
    sesion.cerrar()
sesion = SesionETL(RAIZ, modo='manual')
print('Fuentes:', sesion.entrada)
print('Base final de demostración:', sesion.ruta_base)
print('Base final previa existente:', sesion.ruta_base.exists())
print('Todavía no se leyó ningún CSV ni se abrió SQLite.')

## E · Extracción

El código compartido utiliza `pd.read_csv`. Comprueba columnas, conteos y SHA-256, y devuelve los datos crudos como texto para preservar identificadores. Los errores intencionales todavía están presentes.

`extraccion/preparar.py` coordina siete funciones específicas. Cada una lee solamente su CSV; clientes y productos tienen una función adicional para sus cambios. `lector_csv.py` comparte lectura, contratos y huellas. Todas las fuentes actuales siguen siendo CSV locales.

In [ ]:
display(Code(inspect.getsource(extraer_fuentes), language='python'))
display(Code(inspect.getsource(leer_csv_validado), language='python'))
fuentes = sesion.extraer()
display(sesion.manifiesto[['archivo','filas']])
display(fuentes['ventas'].head())
display(fuentes['ventas'][fuentes['ventas']['id_comprobante'].str.startswith('ERROR')])
assert not sesion.ruta_temporal.exists()
print('Extracción lista: datos en memoria; SQLite todavía sin abrir.')

## T.1 · Dimensiones y SCD

Se normalizan claves y correos y se eliminan duplicados idénticos. Tipo 1 corrige nombres; tipo 2 conserva versiones de región/segmento del cliente; tipo 3 conserva categoría/subcategoría actual y anterior. Puedes leer el código de esta función en `src/transformacion/dimensiones/preparar.py`, que llama a `clientes.py`, `productos.py` y `sucursales.py` de esa misma carpeta.

El calendario se prepara después de validar los hechos, en `dimensiones/fechas.py`, porque su período usa ventas y devoluciones aceptadas. La limpieza de correos sigue comprobando vacíos y duplicados; no valida su formato.

In [ ]:
sucursales, productos, clientes = sesion.transformar_dimensiones()
display(fuentes['productos'].head(1))
display(productos.head(1))
display(clientes[clientes['id_cliente'].eq('C000001')])
print('Personas sintéticas:', clientes['id_cliente'].nunique(), '| Versiones SCD:', len(clientes))

## T.2 · Ventas

Se validan fechas, claves, cantidades, descuentos e importes. Las ventas reciben las claves del modelo y la versión del cliente vigente al vender. La fila rechazada conserva su registro original y un motivo principal.

In [ ]:
ventas = sesion.transformar_ventas()
display(ventas[['clave_negocio','fecha_venta','clave_cliente','clave_producto',
                 'clave_sucursal_canal','importe_bruto','importe_descuento','importe_tras_descuento']].head())
display(pd.DataFrame(sesion.contexto.rechazos)[['fuente','clave_origen','motivo']])
print('Ventas leídas:', len(fuentes['ventas']), '| Aceptadas:', len(ventas))

## T.3 · Devoluciones

Cada evento debe corresponder a una venta aceptada. Se comprueban fecha y cantidades e importes proporcionales acumulados. Las dimensiones comerciales corresponden a la venta original y la fecha es la de devolución.

In [ ]:
devoluciones = sesion.transformar_devoluciones()
display(devoluciones.head())
display(pd.DataFrame(sesion.contexto.rechazos).query("fuente == 'devoluciones'")[['clave_origen','motivo']])
print('Devoluciones aceptadas:', len(devoluciones))

## T.4 · Conciliación y resultado limpio

Se construye el calendario. Cada fuente cumple: leídas = aceptadas + rechazadas + duplicadas. Aquí las seis tablas están preparadas en memoria. En las dos siguientes celdas se guardarán en Parquet y se volverán a leer para cargar Gold. También se exporta un CSV fijo de rechazos.

In [ ]:
tablas, rechazos, duplicados, conciliacion, cambios = sesion.conciliar()
display(conciliacion)
display(pd.DataFrame([{'tabla':nombre,'filas_preparadas':len(datos)} for nombre,datos in tablas.items()]))
assert conciliacion['diferencia'].eq(0).all()
assert not sesion.ruta_temporal.exists()
print('Transformación terminada. Aún no abrimos la base de datos.')

## T.5 · Guardar Silver en Parquet

Se escriben las cuatro dimensiones y los dos hechos limpios usando PyArrow y compresión Snappy. Primero se prepara un conjunto temporal y se valida la lectura de vuelta; después reemplaza la Silver anterior. El manifiesto guarda la ejecución, columnas, tipos, filas y SHA-256 de cada archivo. No se elimina ni convierte el CSV original.

In [ ]:
manifiesto_silver = sesion.guardar_silver()
print('Silver persistida:', sesion.carpeta_silver)
display(pd.DataFrame([
    {'tabla': fila['tabla'], 'archivo': fila['archivo'], 'filas': fila['filas'],
     'bytes_parquet': (sesion.carpeta_silver / fila['archivo']).stat().st_size}
    for fila in manifiesto_silver['tablas']
]))
assert len(list(sesion.carpeta_silver.glob('*.parquet'))) == 6
assert not sesion.ruta_temporal.exists(), 'SQLite todavía no debe existir.' 

## T.6 · Leer Silver: estas filas se cargarán a Gold

El proceso vuelve a abrir los seis archivos Parquet y comprueba el manifiesto y sus huellas. Gold utiliza estas tablas leídas desde disco, no los DataFrames anteriores. Las fechas ISO siguen siendo texto porque ese es el tipo definido en nuestro modelo; Parquet no cambia sus tipos automáticamente.

In [ ]:
display(Code(inspect.getsource(leer_silver), language='python'))
tablas = sesion.leer_silver()
display(tablas['hecho_ventas'].head())
# Ejemplo adicional de lectura selectiva de columnas, sin crear otro archivo.
solo_importes = pd.read_parquet(
    sesion.carpeta_silver / 'hecho_ventas.parquet',
    columns=['clave_venta', 'importe_tras_descuento'], engine='pyarrow'
)
display(solo_importes.head())
assert len(solo_importes) == len(tablas['hecho_ventas'])
assert sesion.resumen['gold_lee_desde_silver'] is True
print('Silver verificada y lista para cargar Gold.')

## L.1 · Conexión real a SQLite

`create_engine` configura la conexión y `motor.connect()` la abre. SQLite usa un archivo local y no necesita servidor ni contraseña. Se cargan los DataFrames leídos desde Parquet. SQLite se construye primero en un archivo temporal fijo; la base final anterior permanece disponible.

In [ ]:
display(Code(inspect.getsource(crear_motor), language='python'))
motor = sesion.conectar()
print('URL SQLAlchemy:', motor.url)
print('Base temporal para inspeccionar durante la carga:', sesion.ruta_temporal)
print('Base final que se reemplazará al terminar:', sesion.ruta_base)
display(sesion.consultar("SELECT name AS tabla FROM sqlite_master WHERE type='table'"))

## L.2 · Modelo vacío

Se ejecuta el SQL en español. Las claves primarias, foráneas y restricciones protegen el modelo. Puedes abrir la base temporal en DB Browser para mostrar las tablas vacías; cierra el visor antes de escribir de nuevo.

In [ ]:
for archivo_sql in ARCHIVOS_MODELO:
    print('SQL:', archivo_sql)
    display(Code((RAIZ / 'sql' / archivo_sql).read_text(encoding='utf-8'), language='sql'))
sesion.crear_modelo()
display(sesion.consultar("SELECT name AS tabla FROM sqlite_master WHERE type='table' ORDER BY name"))
display(sesion.consultar('SELECT COUNT(*) AS ventas_cargadas FROM hecho_ventas'))

## L.3 · Cargar dimensiones

Se insertan primero porque las claves de los hechos deben apuntar a dimensiones existentes. La inserción real es `DataFrame.to_sql(..., conexion, if_exists='append')`; append se aplica a la base temporal nueva, no acumula filas en la base final.

In [ ]:
display(Code(inspect.getsource(cargar_tablas), language='python'))
sesion.cargar_dimensiones()
display(sesion.consultar('SELECT COUNT(*) AS versiones_cliente FROM dim_cliente'))
display(sesion.consultar('SELECT COUNT(*) AS ventas_todavia_vacias FROM hecho_ventas'))

## L.4 · Cargar hechos y trazabilidad

Se insertan ventas, luego devoluciones y finalmente auditoría. Los duplicados, cambios y conciliaciones se guardan dentro de SQLite para evitar archivos adicionales.

In [ ]:
sesion.cargar_hechos()
display(sesion.consultar("SELECT 'ventas' AS tabla, COUNT(*) AS filas FROM hecho_ventas UNION ALL SELECT 'devoluciones', COUNT(*) FROM hecho_devoluciones"))
display(sesion.consultar('SELECT * FROM etl_conciliacion'))

## L.5 · Validar lo cargado

Los controles y métricas se calculan mediante consultas sobre SQLite. No se publica una base con controles fallidos.

In [ ]:
controles = sesion.validar()
display(pd.DataFrame([{'control':nombre,'infracciones':valor} for nombre,valor in controles.items()]))
display(pd.Series(sesion.resumen['metricas']).to_frame('valor'))
assert not any(controles.values())

## Publicar: reemplazar la base final

Solo ahora la base validada reemplaza `resultados/demostracion.sqlite`. Si falla antes, se elimina la temporal y se conserva la base anterior. La sesión se cierra y se libera el bloqueo.

In [ ]:
resumen = sesion.publicar()
assert resumen['estado'] == 'CORRECTA'
assert not sesion.ruta_temporal.exists()
print('Carga manual terminada:', sesion.ruta_base)
print('Ejecución:', resumen['id_ejecucion'])
print('Resumen y log sobrescritos en resultados, sin carpetas por ejecución.')

## Consultar la base final y mostrarla al profesor

Abre exactamente la ruta impresa arriba en DB Browser. Puedes usar las mismas consultas SQL de esta celda. La ejecución completa crea `dataretail360.sqlite`; este notebook crea `demostracion.sqlite`. Ambos contienen el mismo modelo y reglas.

In [ ]:
motor_lectura = crear_motor(sesion.ruta_base)
try:
    with motor_lectura.connect() as conexion:
        display(pd.read_sql_query('SELECT * FROM hecho_ventas LIMIT 5', conexion))
        display(pd.read_sql_query('SELECT * FROM vista_actividad_mensual ORDER BY anio,mes,canal LIMIT 8', conexion))
        display(pd.read_sql_query("SELECT * FROM dim_cliente WHERE id_cliente='C000001' ORDER BY vigente_desde", conexion))
        display(pd.read_sql_query('SELECT * FROM vista_detalle_devoluciones LIMIT 5', conexion))
        display(pd.read_sql_query('SELECT codigo_producto,nombre_producto,unidades_netas,ventas_netas FROM vista_ventas_por_producto ORDER BY ventas_netas DESC LIMIT 5', conexion))
        display(pd.read_sql_query('SELECT nombre_sucursal,canal,ventas_netas,comprobantes,ticket_promedio_antes_devoluciones FROM vista_ventas_por_sucursal ORDER BY ventas_netas DESC', conexion))
finally:
    motor_lectura.dispose()

## Repetir y limpiar

Para repetir: reinicia el kernel y vuelve a ejecutar desde la primera celda. Las fuentes se reutilizan y las salidas fijas se reemplazan. Para comenzar sin bases: cierra DB Browser, detén o reinicia el kernel para liberar cualquier sesión y ejecuta `LIMPIAR_RESULTADOS.cmd`.

No borres `datos/entrada`, la carpeta `sql`, `requirements.txt` ni `.venv`. La limpieza controlada conserva esos archivos. Si abandonas una sesión antes de publicar, puedes cerrarla explícitamente ejecutando `sesion.cerrar()` en una celda; no publica la base parcial.